# small128 iter-1 — first improvement-loop iteration on the 4× small model

**Base (warm-start): `pillar3k_small128_hardce_epoch_87`** — the distilled 10b×128ch student.
Bar on held-out 775000-775499 (500 seeds, fp16): **mean 12,987 / P50 8,889 / P10 1,799 / <1000 4.8%**.

**Corpus `small128_iter1.pt` = 2,974,799 states (75% crisis / 25% selfplay), ALL C++-mined from ep87 itself:**
- crisis_cpp128_v1: 7,275 escape-or-die replays of ep87's own greedy deaths
  (recovery 15-back @2400 sims, 36% escape; prevention 50-back @1600, 82% escape). 2.24M states.
- selfplay_cpp128_v2: 500 fresh ep87 selfplay games @1600 sims (cap 1500). 0.73M states.
- Decisiveness: **33.9% / 29.7% of states >0.40 visit top-share** (pillar3k-era corpus: 15-22%).

**Recipe = the pillar3k unlock (HISTORY 173-174):** decisiveness-weighted CE (`--decisiveness-power 3.0`)
+ `--target-temperature 0.7`, warm-start, judged by GAMEPLAY FLOOR.
- **VAL IS DOUBLY UNRELIABLE under dw** (rose while gameplay +18% on pillar3k). Ignore it.
- **Pick a MID-TRAINING epoch by floor** (P10 / <1000): pillar3k's P50 peaked ep17, P10 ep22, both declined by ep27.
- Prior failure to beat: sp1 (100%-selfplay corpus from a stale model) regressed monotonically. This run
  fixes both causes (right teacher data + 75% crisis). If floor regresses monotonically ep1→ep4 anyway,
  STOP and drop LR to 1e-4 (pillar3i lesson) for a re-run.

**Upload to Drive `MyDrive/alphatrain/`:** `small128_iter1.pt.gz` (231,393,038 B) — tarball
`colorlines_pillar3d_v2.tar.gz` + `pillar3k_small128_hardce_epoch_87.pt` are already there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/small128_iter1.pt.gz /content/small128_iter1.pt.gz
gz=os.path.getsize('/content/small128_iter1.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 231_393_038, f'.gz truncated! got {gz}; re-upload small128_iter1.pt.gz'
!gunzip -t /content/small128_iter1.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/small128_iter1.pt.gz > /content/alphatrain/data/small128_iter1.pt
pt=os.path.getsize('/content/alphatrain/data/small128_iter1.pt')
assert pt == 1_139_353_549, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 2,974,799 states ({time.time()-t0:.0f}s)')
!rm /content/small128_iter1.pt.gz
!cp {DRIVE}/pillar3k_small128_hardce_epoch_87.pt /content/alphatrain/data/
print('base ckpt:', os.path.getsize('/content/alphatrain/data/pillar3k_small128_hardce_epoch_87.pt'), 'bytes')
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (the pillar3k recipe, adapted to the 128ch base) =====
CHANNELS = 128
EPOCHS   = 28        # pillar3k v2 trained 28, best ep22 (picked by floor)
BATCH    = 32768     # ~726 steps/epoch on 23.8M effective samples (aug x8)
LR       = 3e-4      # the EXACT pillar3k value (worked warm-started WITH dw3).
                     # FALLBACK: if floor regresses monotonically ep1->ep4, re-run with 1e-4.
DW       = 3.0       # decisiveness power: decisive escapes drive the gradient
T        = 0.7       # target temperature (pillar3k operating point)
RUN      = "small128_iter1"
print(f'RUN={RUN}  ch={CHANNELS} epochs={EPOCHS} batch={BATCH} lr={LR} dw={DW} T={T}')

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/small128_iter1.pt \
    --resume alphatrain/data/pillar3k_small128_hardce_epoch_87.pt --warm-start \
    --channels {CHANNELS} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log
# REMINDER: val is expected to look bad/noisy under decisiveness weighting. Gameplay is the truth.

In [ ]:
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Eval — on the M5, with the C++ engine (Python is retired for eval)

Per candidate epoch (mid-training epochs first: ~14/18/22/26):
```bash
# once per checkpoint: export + verify it's the right model
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/small128_iter1_epoch_22.pt
cd alphatrain/inference_cpp
./build/eval --model data/policy_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
**Bar = ep87: mean 12,987 / P50 8,889 / P10 1,799 / <1000 4.8%** (same seed range; compare
distributions, never per-seed).

- Judge by **floor first** (P10, <1000), then median. Mean is tail-noisy.
- Candidates that clear the bar on 500 seeds → decide between them on the **5k range**
  (775000-780000; 500 seeds is too noisy for close calls — pillar3k lesson 6).
- Success = first loop iteration that BEATS its base. Then: mine crisis on the NEW model → iterate.
- Monotonic floor regression from ep1 → stop, re-run at LR=1e-4.